# Evaluation: Food Decision Intelligence (methodology v0.1.0)

This notebook runs every evaluation the available labelled data allows and states plainly what could not run.
**Read the sample sizes.** The labelled sets are small, team-labelled and partly synthetic
(`data/verified/eval/README.md`), so these numbers show that the evaluation code works and give a first signal.
They are not accuracy claims about real products. The build spec's target of a 100–200 product verified set is a
target, not a result.

Run from the repository root: `jupyter nbconvert --to notebook --execute notebooks/evaluation.ipynb`.
The same code runs headless with `PYTHONPATH=src python -m evaluation.run`.

In [1]:
import sys, json, pathlib
ROOT = pathlib.Path.cwd() if (pathlib.Path.cwd() / 'src').exists() else pathlib.Path.cwd().parent
sys.path.insert(0, str(ROOT / 'src'))
import pandas as pd
pd.set_option('display.precision', 3)
from evaluation import run as ev
results = ev.run_all(write=True)
print('methodology', results['methodology_version'])

methodology 0.1.0


## 1. Extraction: precision, recall, exact-match accuracy
Field-level comparison against hand-read values for the four synthetic test pages. Numeric values match within 2 %. The fourth page has no structured data, so with no LLM key the deterministic extractors find nothing there; that page is what the optional LLM fallback is for.

In [2]:
e = results['extraction']
print(e['data']); print('pages:', e['n_pages'], ' fields:', e['overall']['n_fields'])
display(pd.DataFrame([e['overall']]))
display(pd.DataFrame(e['per_page']).T[['n_fields','precision','recall','exact_match_accuracy']])
display(pd.DataFrame(e['per_field']).T[['precision','recall','exact_match_accuracy']])

4 synthetic test pages (tests/fixtures), hand-labelled; deterministic extractors only (no LLM key)
pages: 4  fields: 60


,n_fields,tp,fp,fn,precision,recall,exact_match_accuracy
0,60,38,1,10,0.974,0.792,0.833


,n_fields,precision,recall,exact_match_accuracy
cereal_jsonld.html,15.0,1.000,1.000,1.000
biscuit_html_only.html,15.0,0.929,0.929,0.933
noodles_per_serving.html,15.0,1.000,1.000,1.000
beverage_unstructured.html,15.0,NaN,0.000,0.400


,precision,recall,exact_match_accuracy
name,0.667,0.50,0.50
brand,1.000,0.75,0.75
pack_size,1.000,0.75,0.75
pack_unit,1.000,0.75,0.75
price_inr,1.000,0.75,0.75
energy,1.000,0.75,0.75
protein,1.000,0.75,0.75
carbohydrate,1.000,1.00,1.00
total_sugars,1.000,0.75,0.75
added_sugars,1.000,1.00,1.00


## 2. Ingredient classification: precision, recall, F1
Top class per ingredient phrase against team labels. `uncertain` means the system declined to classify; it counts as no label (a missed class is a false negative, not a false positive). **Caveat:** the same team wrote the ingredient dictionary and these labels, so a perfect score here mostly shows the dictionary and parser agree with the team's own definitions; it says little about unseen real labels.

In [3]:
i = results['ingredients']
print(i['data']); print('items:', i['n_items'])
display(pd.DataFrame([i['classification']]))
print('uncertainty flag (expected uncertain vs flagged uncertain):')
display(pd.DataFrame([i['uncertainty_flag']]))
print('errors:', i['errors'] or 'none')

team-labelled ingredient phrases (data/verified/eval/ingredient_gold.csv)
items: 50


,tp,fp,fn,precision,recall,f1
0,43,0,0,1.0,1.0,1.0


uncertainty flag (expected uncertain vs flagged uncertain):


,tp,fp,fn,precision,recall,f1,tn,false_positive_rate
0,7,0,0,1.0,1.0,1.0,43,0.0


errors: none


## 3. Claim checks: precision, recall, false-positive rate
Expected statuses were worked out by hand from the FSSAI conditions before running the code. The positive class is *potential inconsistency for review*. **Caveat:** 22 hand-built cases test that the rules are implemented as written; they do not measure performance on real listings, where claim wording and missing data vary far more.

In [4]:
c = results['claims']
print(c['data']); print('cases:', c['n_cases'], ' status accuracy:', round(c['status_accuracy'], 3))
display(pd.DataFrame([c['inconsistency_detection']]))
print('disagreements:', c['disagreements'] or 'none')

hand-worked cases from FSSAI Schedule I, reg. 6 and Schedule V (data/verified/eval/claim_gold.csv)
cases: 22  status accuracy: 1.0


,tp,fp,fn,precision,recall,f1,tn,false_positive_rate
0,8,0,0,1.0,1.0,1.0,14,0.0


disagreements: none


## 4. Product matching: correct-match rate
The matcher links a new listing to a stored Open Food Facts record (reference only). Queries are perturbed names of the synthetic sample plus products that are not stored (correct answer: no match).

In [5]:
m = results['product_matching']
print(m['data']); print('queries:', m['n_queries'], ' correct-match rate:', round(m['correct_match_rate'], 3))
display(pd.DataFrame(m['by_perturbation']).T)

40 synthetic sample records with generated name perturbations; near-identical names ('Sample Cereal A' vs 'B') make this a hard case
queries: 90  correct-match rate: 0.989


,n,correct_rate
case and order,40.0,1.000
typo,40.0,0.975
absent product,10.0,1.000


## 5. Scoring against Nutri-Score and expert rankings

In [6]:
print(json.dumps(results['scoring_vs_nutriscore'], indent=1))
print(json.dumps(results['expert_ranking'], indent=1))

{
 "status": "not run",
 "reason": "needs Open Food Facts records with nutriscore_grade (data/raw/off_india.jsonl from ./run.sh off); the OFF API was not reachable from the build environment"
}
{
 "status": "not run",
 "reason": "no expert rankings collected yet"
}


## 6. Recommendations: Precision@3 and NDCG@5

In [7]:
print(json.dumps(results['recommendations'], indent=1))
from evaluation.metrics import precision_at_k, ndcg_at_k
# The metric functions are ready; example on a toy ranking (not a result):
print('toy example only:', precision_at_k(['a','b','c'], {'a','c'}, 3), round(ndcg_at_k(['b','a'], {'a':3,'b':1}), 3))

{
 "status": "not run",
 "reason": "Precision@3 and NDCG@5 need relevance judgements for alternatives; none collected yet (metric code: evaluation/metrics.py)"
}
toy example only: 0.6666666666666666 0.797


## 7. Robustness
**Weight sensitivity:** each dimension weight scaled by 0.8 or 1.2 (then renormalised). **Missing fields:** one nutrient removed at a time; *overall_not_computed* counts products whose nutrition coverage fell below 50 %, so no grade is given instead of a guessed one. **Conflicting sources:** a second source with changed values must be flagged when the difference is over 2 %.

In [8]:
r = results['robustness']
print(r['weights']['data']); display(pd.DataFrame(r['weights']['perturbations']).T)
print(r['missing_fields']['data']); display(pd.DataFrame(r['missing_fields']['by_nutrient']).T)
print(r['conflicting_sources']['data'], '- cases:', r['conflicting_sources']['n_cases'])
display(pd.DataFrame([r['conflicting_sources']['detection']]))

40 synthetic sample records


,n,grade_changed,mean_abs_score_change
nutrition x0.8,38.0,4.0,1.34
nutrition x1.2,38.0,3.0,1.18
ingredients x0.8,38.0,4.0,0.71
ingredients x1.2,38.0,0.0,0.63
transparency x0.8,38.0,5.0,1.05
transparency x1.2,38.0,2.0,0.92
claims x0.8,38.0,0.0,0.74
claims x1.2,38.0,2.0,0.76
value x0.8,38.0,1.0,0.37
value x1.2,38.0,2.0,0.37


40 synthetic sample records; one nutrient removed at a time


,n,overall_not_computed,mean_abs_score_change
total_sugars,37.0,10.0,2.41
saturated_fat,36.0,7.0,3.41
sodium,38.0,6.0,2.12
fibre,26.0,1.0,3.48
protein,38.0,0.0,3.63
total_fat,38.0,0.0,1.61


cereal test page with a second, modified source - cases: 6


,tp,fp,fn,precision,recall,f1,tn,false_positive_rate
0,3,0,0,1.0,1.0,1.0,3,0.0


## 8. Targets not yet met

In [9]:
for t in results['targets_not_yet_met']: print('-', t)

- verified evaluation set of 100-200 real products (build spec target)
- 500-1,000 Open Food Facts India records (needs ./run.sh off with network access)
